In [1]:
import geopandas as gpd
import Levenshtein
import numpy as np
import pandas as pd
from shapely.geometry import Point

import os
import sys

# use absolute path here
project_path = "/mnt/School/PhD/AI221/Project/"
sys.path.insert(0, project_path)

from src.data_extraction.utils.constants import data_path

In [2]:
fcp_path = os.path.join(data_path, "flood_control_projects")

In [3]:
fcp_df = pd.read_csv(
    os.path.join(fcp_path, "flood_control_raw.csv")
)
fcp_df.head()

,InfraYear,Region,Province,Municipality,ImplementingOffice,ProjectID,ProjectDescription,ProjectComponentID,ProjectComponentDescription,Program,...,EditDate,Editor,FundingYear,LegislativeDistrict,DistrictEngineeringOffice,GlobalID,ABC_String,ContractCost_String,CompletionDateActual,StartDate
0,2025,Region I,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),Pangasinan 1st District Engineering Office,P00941153LZ,"Rehabilitation of Flood Mitigation Structure, ...",P00941153LZ-CW1,Rehabilitation / Major Repair of Flood Control...,NaN,...,1754303523936,dpwh_view,2025,PANGASINAN (FIRST LEGISLATIVE DISTRICT),Pangasinan 1st District Engineering Office,dbb53f51-7acf-4ab1-9e48-7c7f5b0fc3e5,4950000,4850385.71,2025-05-14,02/21/2025
1,2025,Region IV-B,PALAWAN,PUERTO PRINCESA CITY (CAPITAL) (PALAWAN),Palawan 3rd District Engineering Office,P00921211LZ,"Construction of Slope Protection Structure, In...",P00921211LZ-CW1,Construction of Flood Mitigation Structure - C...,NaN,...,1754303523936,dpwh_view,2025,PALAWAN (THIRD LEGISLATIVE DISTRICT),Palawan 3rd District Engineering Office,53062625-c774-40c1-a14f-6831eb68eca4,14700000,14669999.48,2025-05-19,02/05/2025
2,2025,Region I,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),Pangasinan 1st District Engineering Office,P00941152LZ,"Construction of Line Canal, Barangay Quibuar, ...",P00941152LZ-CW1,Construction of Flood Mitigation Structure - C...,NaN,...,1754303523936,dpwh_view,2025,PANGASINAN (FIRST LEGISLATIVE DISTRICT),Pangasinan 1st District Engineering Office,fb77172b-e743-42dc-8dd5-ea2bb9744920,4950000,4850666.92,2025-05-21,02/10/2025
3,2025,Region III,TARLAC,CITY OF TARLAC (CAPITAL) (TARLAC),Tarlac District Engineering Office,P00921064LZ,Construction of Slope Protection Structure alo...,P00921064LZ-CW1,Construction of Slope Protection Structure - C...,NaN,...,1754303523936,dpwh_view,2025,TARLAC (FIRST LEGISLATIVE DISTRICT),Tarlac District Engineering Office,b69a08c1-ec48-4629-ae07-134dd733d5c9,14700000,14229719.67,2025-05-22,01/27/2025
4,2025,Region III,TARLAC,MONCADA (TARLAC),Tarlac District Engineering Office,P00921065LZ,Rehabilitation of Sabo Dam at Camangaan West C...,P00921065LZ-CW1,Rehabilitation / Major Repair of Slope Protect...,NaN,...,1754303523936,dpwh_view,2025,TARLAC (FIRST LEGISLATIVE DISTRICT),Tarlac District Engineering Office,75211869-ba47-4f67-acf0-750ee508ea47,14700000,14700000,2025-04-16,01/27/2025


In [4]:
ph_bounds = gpd.read_file(os.path.join(data_path,"ph_adm3_municities/PH_Adm3_MuniCities.shp.shp"))
ph_bounds = ph_bounds[ph_bounds["geo_level"] == "City"]
ph_bounds.head()

,adm1_psgc,adm2_psgc,adm3_psgc,adm3_en,geo_level,len_crs,area_crs,len_km,area_km2,geometry
4,100000000,102800000,102805000,City of Batac,City,66661,158252391,66,158.0,"POLYGON ((247341.309 2003933.537, 247293.327 2..."
11,100000000,102800000,102812000,City of Laoag,City,53964,110146974,53,110.0,"POLYGON ((248393.247 2016552.78, 248424.831 20..."
28,100000000,102900000,102906000,City of Candon,City,62247,77652664,62,77.0,"POLYGON ((230319.064 1907309.065, 230338.289 1..."
56,100000000,102900000,102934000,City of Vigan,City,25067,24485368,25,24.0,"POLYGON ((221597.447 1945779.016, 221718.087 1..."
70,100000000,103300000,103314000,City of San Fernando,City,54233,99006121,54,99.0,"POLYGON ((225191.401 1841887.86, 225339.01 184..."


# Preprocessing

In [5]:
provincial_df = pd.read_csv(os.path.join(fcp_path, "PH_Adm2_ProvDists.csv"))
provincial_df.head()

,adm1_psgc,adm2_psgc,adm2_en,geo_level,len_crs,area_crs,len_km,area_km2
0,100000000,102800000,Ilocos Norte,Prov,309785,3276945154,309,3276.0
1,100000000,102900000,Ilocos Sur,Prov,452374,2467458323,452,2467.0
2,100000000,103300000,La Union,Prov,262415,1414080983,262,1414.0
3,100000000,105500000,Pangasinan,Prov,789136,5161200257,789,5161.0
4,200000000,200900000,Batanes,Prov,230060,201280837,230,201.0


In [6]:
ph_muni_prov_remap = {
    # Conform to the FCP province tag
    # Source: PSGC-2Q-2023-Publication-Datafile.xlsx
    'City of Angeles': 305400000,
    'City of Olongapo': 307100000,
    'City of Lucena': 405600000,
    'City of Bacolod': 604500000,
    'City of Iloilo': 603000000,
    'City of Cebu': 702200000,
    'City of Lapu-Lapu': 702200000,
    'City of Mandaue': 702200000,
    'City of Tacloban': 803700000,
    'City of Zamboanga': 907300000,
    'City of Cagayan De Oro': 1004200000,
    'City of Iligan': 1903600000,
    'City of Davao': 1102400000,
    'City of General Santos': 1206300000,
    'City of Baguio': 1401100000,
    'City of Butuan': 1600200000,
    'City of Puerto Princesa': 1705300000,
    'City of Iligan': 1003500000,
    'City of Cagayan De Oro' : 1004300000,
}

In [7]:
ph_muni_gdf =  ph_bounds[[
        "adm2_psgc",
        "adm3_en",
        "adm3_psgc",
]]
ph_muni_gdf["adm3_psgc"] = ph_muni_gdf["adm3_psgc"].astype(int)
ph_muni_gdf["adm2_psgc"] = ph_muni_gdf.apply(lambda row: ph_muni_prov_remap.get(row["adm3_en"], row["adm2_psgc"]), axis=1)
ph_muni_gdf.head()

,adm2_psgc,adm3_en,adm3_psgc
4,102800000,City of Batac,102805000
11,102800000,City of Laoag,102812000
28,102900000,City of Candon,102906000
56,102900000,City of Vigan,102934000
70,103300000,City of San Fernando,103314000


In [8]:
ph_muni_gdf = pd.merge(
    provincial_df,
    ph_muni_gdf,
    on="adm2_psgc",
    how="outer",
    indicator=True
)
ph_muni_gdf["adm3_psgc"] = ph_muni_gdf["adm3_psgc"].apply(lambda x: pd.NA if np.isnan(x) else int(x))
ph_muni_gdf.head()

,adm1_psgc,adm2_psgc,adm2_en,geo_level,len_crs,area_crs,len_km,area_km2,adm3_en,adm3_psgc,_merge
0,100000000.0,102800000,Ilocos Norte,Prov,309785.0,3.276945e+09,309.0,3276.0,City of Batac,102805000,both
1,100000000.0,102800000,Ilocos Norte,Prov,309785.0,3.276945e+09,309.0,3276.0,City of Laoag,102812000,both
2,100000000.0,102900000,Ilocos Sur,Prov,452374.0,2.467458e+09,452.0,2467.0,City of Candon,102906000,both
3,100000000.0,102900000,Ilocos Sur,Prov,452374.0,2.467458e+09,452.0,2467.0,City of Vigan,102934000,both
4,100000000.0,103300000,La Union,Prov,262415.0,1.414081e+09,262.0,1414.0,City of San Fernando,103314000,both


In [9]:
# Validation: only metro manila cities remain unmatched
ph_muni_gdf[ph_muni_gdf["_merge"] == "right_only"]

,adm1_psgc,adm2_psgc,adm2_en,geo_level,len_crs,area_crs,len_km,area_km2,adm3_en,adm3_psgc,_merge
141,NaN,1380100000,NaN,NaN,NaN,NaN,NaN,NaN,City of Caloocan,1380100000,right_only
142,NaN,1380200000,NaN,NaN,NaN,NaN,NaN,NaN,City of Las Piñas,1380200000,right_only
143,NaN,1380300000,NaN,NaN,NaN,NaN,NaN,NaN,City of Makati,1380300000,right_only
144,NaN,1380400000,NaN,NaN,NaN,NaN,NaN,NaN,City of Malabon,1380400000,right_only
145,NaN,1380500000,NaN,NaN,NaN,NaN,NaN,NaN,City of Mandaluyong,1380500000,right_only
146,NaN,1380600000,NaN,NaN,NaN,NaN,NaN,NaN,City of Manila,1380600000,right_only
147,NaN,1380700000,NaN,NaN,NaN,NaN,NaN,NaN,City of Marikina,1380700000,right_only
148,NaN,1380800000,NaN,NaN,NaN,NaN,NaN,NaN,City of Muntinlupa,1380800000,right_only
149,NaN,1380900000,NaN,NaN,NaN,NaN,NaN,NaN,City of Navotas,1380900000,right_only
150,NaN,1381000000,NaN,NaN,NaN,NaN,NaN,NaN,City of Parañaque,1381000000,right_only


In [10]:
# City of Isabela can be tagged us under Basilan (https://en.wikipedia.org/wiki/Isabela,_Basilan)
# Adm1 corresponds to: Region IX (https://github.com/altcoder/philippines-psgc-shapefiles/blob/main/dist/PH_Adm1_Regions.csv)
ph_muni_gdf.loc[
    ph_muni_gdf.adm3_psgc == 990101000,
    "adm2_psgc"
] = 1900700000
ph_muni_gdf.loc[
    ph_muni_gdf.adm3_psgc == 990101000,
    "adm2_en"
] = "Basilan"

In [11]:
ph_muni_gdf_final = ph_muni_gdf.loc[
    ph_muni_gdf._merge.isin(["both", "right_only"]), 
    ["adm2_en", "adm3_psgc", "adm3_en"]
]
ph_muni_gdf_final["adm2_en"] = ph_muni_gdf_final["adm2_en"].fillna(ph_muni_gdf_final["adm3_en"])
ph_muni_gdf_final.head()

,adm2_en,adm3_psgc,adm3_en
0,Ilocos Norte,102805000,City of Batac
1,Ilocos Norte,102812000,City of Laoag
2,Ilocos Sur,102906000,City of Candon
3,Ilocos Sur,102934000,City of Vigan
4,La Union,103314000,City of San Fernando


In [12]:
fcp_gdf = gpd.GeoDataFrame(fcp_df)

In [13]:
# Cleaning funcs to conform to the PSGC dataset
def clean_province(
    province_str: str,
    muni_str: str,
) -> str:
    province_str = province_str.strip()

    # handle metro manila
    if " CITY" in province_str:
        if (
            province_str == "MUNTINLUPA CITY" 
            and muni_str == "CITY OF LAS PIÑAS (METROPOLITAN MANILA)"
        ):
            province_str = "CITY OF LAS PIÑAS"
        elif province_str not in [
            "QUEZON CITY",
            "PASAY CITY"
        ]:
            province_str = "CITY OF " + province_str.replace(" CITY", "")
        
    province_map = {
        "SAMAR (WESTERN SAMAR)": "SAMAR",
        "COTABATO (NORTH COTABATO)": "COTABATO",
    }
    return province_map.get(province_str, province_str)


def clean_municipality(muni_str: str) -> str:
    muni_str = muni_str.strip()

    city_map = {
        "CITY OF BALIUAG" : "CITY OF BALIwAG",
        "CITY OF SANTO TOMAS": "CITY OF STO. TOMAS",
    }
    city_suffix_cases = [
    ] 
    city_prefix_cases = [ 
        "PUERTO PRINCESA CITY",
        "CABANATUAN CITY",
        "CAVITE CITY",
        "TRECE MARTIRES CITY",
        "LA CARLOTA CITY",
        "GENERAL SANTOS CITY",
        "SAN CARLOS CITY",
        "TAGBILARAN CITY",
        "BUTUAN CITY",
        "OLONGAPO CITY",
        "BAGUIO CITY",
        "CABUYAO CITY",
        "NAGA CITY",
        "CADIZ CITY",
        "BAIS CITY",
        "ILOILO CITY",
        "PAGADIAN CITY",
        "GINGOOG CITY",
        "SURIGAO CITY",
        "TAGAYTAY CITY",
        "IRIGA CITY",
        "DAPITAN CITY",
        "TACURONG CITY",
        "COTABATO CITY",
        "ROXAS CITY",
        "MANDAUE CITY",
        "TAGUIG CITY",
        "DAGUPAN CITY",
        "ANGELES CITY",
        "LAOAG CITY",
        "IMUS CITY",
        "BACOOR CITY",
        "SAN PABLO CITY",
        "PALAYAN CITY",
        "LIPA CITY",
        "ZAMBOANGA CITY",
        "LUCENA CITY",
        "ILAGAN CITY",
        "SILAY CITY",
        "CANLAON CITY",
        "SAGAY CITY",
        "BACOLOD CITY",
        "TANGUB CITY",
        "LEGAZPI CITY",
        "TOLEDO CITY",
        "CEBU CITY",
        "CALBAYOG CITY",
        "CAGAYAN DE ORO CITY",
        "ILIGAN CITY",
        "TACLOBAN CITY",
        "DAVAO CITY",
        "PANABO CITY",
        "BAGO CITY",
        "DUMAGUETE CITY",
        "LAPU-LAPU CITY",
        "TAGUIG CITY",
        "MARAWI CITY",
    ]
    
    if muni_str in city_suffix_cases:
        return muni_str.replace("CITY OF", "") + " CITY"
    elif muni_str in city_prefix_cases:
        return "CITY OF " + muni_str.replace(" CITY", "")
    else:
        return city_map.get(muni_str, muni_str)

In [14]:
muni_fcp =  fcp_gdf.loc[
    fcp_gdf["Municipality"].notna(),
    ["Province", "Municipality"]
].drop_duplicates()

muni_fcp["Province_clean"] = muni_fcp.apply(lambda row: 
    clean_province(row["Province"], row["Municipality"]),
    axis=1
)
muni_fcp["Municipality_clean"] = muni_fcp["Municipality"].apply(lambda x: str(x).split("(")[0])
muni_fcp["Municipality_clean"] = muni_fcp["Municipality_clean"].apply(clean_municipality)

## Getting city-only FCP records

In [15]:
# Init: Cross-join all Province-Municipality pairs
muni_cross_df = pd.merge(muni_fcp, ph_muni_gdf_final, how='cross')

# Cast columns to lowercase to make similarity scores more accurate
for col in muni_cross_df:
    if col == "adm3_psgc":
        continue
    muni_cross_df[f"lower_{col}"] = muni_cross_df[col].str.lower()
    
muni_cross_df.head()

,Province,Municipality,Province_clean,Municipality_clean,adm2_en,adm3_psgc,adm3_en,lower_Province,lower_Municipality,lower_Province_clean,lower_Municipality_clean,lower_adm2_en,lower_adm3_en
0,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),PANGASINAN,CITY OF ALAMINOS,Ilocos Norte,102805000,City of Batac,pangasinan,city of alaminos (pangasinan),pangasinan,city of alaminos,ilocos norte,city of batac
1,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),PANGASINAN,CITY OF ALAMINOS,Ilocos Norte,102812000,City of Laoag,pangasinan,city of alaminos (pangasinan),pangasinan,city of alaminos,ilocos norte,city of laoag
2,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),PANGASINAN,CITY OF ALAMINOS,Ilocos Sur,102906000,City of Candon,pangasinan,city of alaminos (pangasinan),pangasinan,city of alaminos,ilocos sur,city of candon
3,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),PANGASINAN,CITY OF ALAMINOS,Ilocos Sur,102934000,City of Vigan,pangasinan,city of alaminos (pangasinan),pangasinan,city of alaminos,ilocos sur,city of vigan
4,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),PANGASINAN,CITY OF ALAMINOS,La Union,103314000,City of San Fernando,pangasinan,city of alaminos (pangasinan),pangasinan,city of alaminos,la union,city of san fernando


In [16]:
def get_muni_similarity_score_pass2(
    muni_str_1: str,
    muni_str_2: str,
) -> float:
    muni_str_1 = muni_str_1.replace("city of ", "").replace(" city", "")
    muni_str_2 = muni_str_2.replace("city of ", "").replace(" city", "")
    return Levenshtein.jaro_winkler(muni_str_1, muni_str_2)

# Sim score between Province strings
muni_cross_df["province_similarity_score"] = muni_cross_df.apply(
    lambda row: Levenshtein.jaro_winkler(row["lower_Province_clean"], row["lower_adm2_en"]),
    axis=1
)

# Sim score between Municipality strings (first pass)
muni_cross_df["muni_similarity_score"] = muni_cross_df.apply(
    lambda row: Levenshtein.jaro_winkler(row["lower_Municipality_clean"], row["lower_adm3_en"]), 
    axis=1
)

# Sim score between Municipality strings without any city prefixes (second pass)
# This is for unmatched rows
mask = muni_cross_df["muni_similarity_score"] < 1
muni_cross_df["muni_similarity_score_pass2"] = pd.NA
muni_cross_df.loc[
    mask, 
    "muni_similarity_score_pass2"
] = muni_cross_df.apply(
    lambda row: get_muni_similarity_score_pass2(row["lower_Municipality_clean"], row["lower_adm3_en"]),
    axis=1
)
muni_cross_df.head()

,Province,Municipality,Province_clean,Municipality_clean,adm2_en,adm3_psgc,adm3_en,lower_Province,lower_Municipality,lower_Province_clean,lower_Municipality_clean,lower_adm2_en,lower_adm3_en,province_similarity_score,muni_similarity_score,muni_similarity_score_pass2
0,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),PANGASINAN,CITY OF ALAMINOS,Ilocos Norte,102805000,City of Batac,pangasinan,city of alaminos (pangasinan),pangasinan,city of alaminos,ilocos norte,city of batac,0.288889,0.878846,0.55
1,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),PANGASINAN,CITY OF ALAMINOS,Ilocos Norte,102812000,City of Laoag,pangasinan,city of alaminos (pangasinan),pangasinan,city of alaminos,ilocos norte,city of laoag,0.288889,0.901282,0.547222
2,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),PANGASINAN,CITY OF ALAMINOS,Ilocos Sur,102906000,City of Candon,pangasinan,city of alaminos (pangasinan),pangasinan,city of alaminos,ilocos sur,city of candon,0.400000,0.894643,0.625
3,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),PANGASINAN,CITY OF ALAMINOS,Ilocos Sur,102934000,City of Vigan,pangasinan,city of alaminos (pangasinan),pangasinan,city of alaminos,ilocos sur,city of vigan,0.400000,0.888549,0.547222
4,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),PANGASINAN,CITY OF ALAMINOS,La Union,103314000,City of San Fernando,pangasinan,city of alaminos (pangasinan),pangasinan,city of alaminos,la union,city of san fernando,0.633333,0.877115,0.541667


In [17]:
# Validation: matching should prioritize Province as there are similar city names
muni_cross_df[
    (muni_cross_df["muni_similarity_score"] == 1)
    & (muni_cross_df["province_similarity_score"] < 1)
][["Province", "Municipality", "adm2_en", "adm3_en"]]

,Province,Municipality,adm2_en,adm3_en
10646,PANGASINAN,SAN CARLOS CITY (PANGASINAN),Negros Occidental,City of San Carlos
10752,LA UNION,CITY OF SAN FERNANDO (CAPITAL) (LA UNION),Pampanga,City of San Fernando
18331,PAMPANGA,CITY OF SAN FERNANDO (CAPITAL) (PAMPANGA),La Union,City of San Fernando
61316,CAMARINES SUR,NAGA CITY (CAMARINES SUR),Cebu,City of Naga
63552,NEGROS OCCIDENTAL,CITY OF TALISAY (NEGROS OCCIDENTAL),Cebu,City of Talisay
68994,NEGROS OCCIDENTAL,SAN CARLOS CITY (NEGROS OCCIDENTAL),Pangasinan,City of San Carlos
73213,CEBU,CITY OF NAGA (CEBU),Camarines Sur,City of Naga
76656,CEBU,CITY OF TALISAY (CEBU),Negros Occidental,City of Talisay
160309,PATEROS,TAGUIG CITY (METROPOLITAN MANILA),City of Taguig,City of Taguig


In [18]:
# Validation: province sim scores < 1 are mismatches
muni_cross_df.loc[muni_cross_df["province_similarity_score"] < 1, ["Province", "adm2_en", "province_similarity_score"]].drop_duplicates()

,Province,adm2_en,province_similarity_score
0,PANGASINAN,Ilocos Norte,0.288889
2,PANGASINAN,Ilocos Sur,0.400000
4,PANGASINAN,La Union,0.633333
9,PANGASINAN,Cagayan,0.671429
10,PANGASINAN,Isabela,0.465079
...,...,...,...
169555,SIQUIJOR,Surigao del Sur,0.519444
169557,SIQUIJOR,Oriental Mindoro,0.416667
169558,SIQUIJOR,Palawan,0.000000
169560,SIQUIJOR,Lanao del Sur,0.467949


In [19]:
# Validation: all of the values that don't have a score of 1 
# are distinct muni/cities and should be discarded
muni_cross_df.loc[
    (muni_cross_df["muni_similarity_score_pass2"].notna()),
    ["Municipality", "lower_Municipality_clean", "lower_adm3_en", 
     "muni_similarity_score", "muni_similarity_score_pass2"]
].drop_duplicates().to_csv("low_muni_sim_score_matches.csv", index=False)

In [20]:
# per Province and Municipality, get the best adm2_en match
muni_cross_df = muni_cross_df[
    muni_cross_df['province_similarity_score'] == muni_cross_df.groupby(
        ['Province', 'Municipality']
    )['province_similarity_score'].transform('max')
]

# per Province and Municipality, get the best adm3_en match
muni_cross_df = muni_cross_df[
    muni_cross_df['muni_similarity_score'] == muni_cross_df.groupby(
        ['Province', 'Municipality']
    )['muni_similarity_score'].transform('max')
]
muni_cross_df.head()

,Province,Municipality,Province_clean,Municipality_clean,adm2_en,adm3_psgc,adm3_en,lower_Province,lower_Municipality,lower_Province_clean,lower_Municipality_clean,lower_adm2_en,lower_adm3_en,province_similarity_score,muni_similarity_score,muni_similarity_score_pass2
5,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),PANGASINAN,CITY OF ALAMINOS,Pangasinan,105503000,City of Alaminos,pangasinan,city of alaminos (pangasinan),pangasinan,city of alaminos,pangasinan,city of alaminos,1.0,1.000000,<NA>
294,PALAWAN,PUERTO PRINCESA CITY (CAPITAL) (PALAWAN),PALAWAN,CITY OF PUERTO PRINCESA,Palawan,1731500000,City of Puerto Princesa,palawan,puerto princesa city (capital) (palawan),palawan,city of puerto princesa,palawan,city of puerto princesa,1.0,1.000000,<NA>
324,TARLAC,CITY OF TARLAC (CAPITAL) (TARLAC),TARLAC,CITY OF TARLAC,Tarlac,306916000,City of Tarlac,tarlac,city of tarlac (capital) (tarlac),tarlac,city of tarlac,tarlac,city of tarlac,1.0,1.000000,<NA>
473,TARLAC,MONCADA (TARLAC),TARLAC,MONCADA,Tarlac,306916000,City of Tarlac,tarlac,moncada (tarlac),tarlac,moncada,tarlac,city of tarlac,1.0,0.535714,0.373016
622,TARLAC,RAMOS (TARLAC),TARLAC,RAMOS,Tarlac,306916000,City of Tarlac,tarlac,ramos (tarlac),tarlac,ramos,tarlac,city of tarlac,1.0,0.423810,0.411111


In [21]:
# Retain fully-matched records
matched_mask = (
    (muni_cross_df["province_similarity_score"] == 1) &
    ((muni_cross_df["muni_similarity_score"] == 1) |
    (muni_cross_df["muni_similarity_score_pass2"] == 1))
)
muni_matched_df = muni_cross_df[matched_mask]

In [22]:
# Adding adm3_psgc to the FCP geodataframe
join_cols = ["Province", "Municipality"] 
muni_cols = join_cols + ["adm3_psgc"]

matched_fcp_gdf = pd.merge(
    fcp_gdf,
    muni_matched_df[muni_cols],
    on=join_cols,
    how="left"
)
matched_fcp_gdf.head()

,InfraYear,Region,Province,Municipality,ImplementingOffice,ProjectID,ProjectDescription,ProjectComponentID,ProjectComponentDescription,Program,...,Editor,FundingYear,LegislativeDistrict,DistrictEngineeringOffice,GlobalID,ABC_String,ContractCost_String,CompletionDateActual,StartDate,adm3_psgc
0,2025,Region I,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),Pangasinan 1st District Engineering Office,P00941153LZ,"Rehabilitation of Flood Mitigation Structure, ...",P00941153LZ-CW1,Rehabilitation / Major Repair of Flood Control...,NaN,...,dpwh_view,2025,PANGASINAN (FIRST LEGISLATIVE DISTRICT),Pangasinan 1st District Engineering Office,dbb53f51-7acf-4ab1-9e48-7c7f5b0fc3e5,4950000,4850385.71,2025-05-14,02/21/2025,105503000
1,2025,Region IV-B,PALAWAN,PUERTO PRINCESA CITY (CAPITAL) (PALAWAN),Palawan 3rd District Engineering Office,P00921211LZ,"Construction of Slope Protection Structure, In...",P00921211LZ-CW1,Construction of Flood Mitigation Structure - C...,NaN,...,dpwh_view,2025,PALAWAN (THIRD LEGISLATIVE DISTRICT),Palawan 3rd District Engineering Office,53062625-c774-40c1-a14f-6831eb68eca4,14700000,14669999.48,2025-05-19,02/05/2025,1731500000
2,2025,Region I,PANGASINAN,CITY OF ALAMINOS (PANGASINAN),Pangasinan 1st District Engineering Office,P00941152LZ,"Construction of Line Canal, Barangay Quibuar, ...",P00941152LZ-CW1,Construction of Flood Mitigation Structure - C...,NaN,...,dpwh_view,2025,PANGASINAN (FIRST LEGISLATIVE DISTRICT),Pangasinan 1st District Engineering Office,fb77172b-e743-42dc-8dd5-ea2bb9744920,4950000,4850666.92,2025-05-21,02/10/2025,105503000
3,2025,Region III,TARLAC,CITY OF TARLAC (CAPITAL) (TARLAC),Tarlac District Engineering Office,P00921064LZ,Construction of Slope Protection Structure alo...,P00921064LZ-CW1,Construction of Slope Protection Structure - C...,NaN,...,dpwh_view,2025,TARLAC (FIRST LEGISLATIVE DISTRICT),Tarlac District Engineering Office,b69a08c1-ec48-4629-ae07-134dd733d5c9,14700000,14229719.67,2025-05-22,01/27/2025,306916000
4,2025,Region III,TARLAC,MONCADA (TARLAC),Tarlac District Engineering Office,P00921065LZ,Rehabilitation of Sabo Dam at Camangaan West C...,P00921065LZ-CW1,Rehabilitation / Major Repair of Slope Protect...,NaN,...,dpwh_view,2025,TARLAC (FIRST LEGISLATIVE DISTRICT),Tarlac District Engineering Office,75211869-ba47-4f67-acf0-750ee508ea47,14700000,14700000,2025-04-16,01/27/2025,NaN


In [23]:
# Validation: Pateros is technically a Municipality and thus is excluded
# source: https://en.wikipedia.org/wiki/Pateros
matched_fcp_gdf.loc[
    (matched_fcp_gdf.adm3_psgc.isna()) &
    (matched_fcp_gdf.Municipality.str.contains("CITY")), 
    ["Province", "Municipality"]
].drop_duplicates()

,Province,Municipality
7097,PATEROS,TAGUIG CITY (METROPOLITAN MANILA)


In [24]:
# Validation: All matched values are cities
matched_fcp_gdf.loc[
    (matched_fcp_gdf.adm3_psgc.notna()) &
    ~(matched_fcp_gdf.Municipality.str.contains("CITY")), 
    ["Province", "Municipality"]
].drop_duplicates()

,Province,Municipality


In [25]:
# Validation: infra_type only has one value
matched_fcp_gdf.infra_type.unique()

<ArrowStringArray>
['Flood Control Structures']
Length: 1, dtype: str

In [26]:
matched_fcp_gdf.ProjectID.value_counts()

ProjectID
P00320211VS    3
P00444687MN    2
P00444655MN    2
P00421301LZ    2
P00421352LZ    2
              ..
P00223240LZ    1
P00223373LZ    1
P00223374LZ    1
P00223238LZ    1
P00223199MN    1
Name: count, Length: 9827, dtype: int64

In [28]:
# Check if there are projects (identified by ProjectID) that has more than 2 records
matched_fcp_gdf.groupby(['ProjectID','InfraYear']).filter(lambda x: len(x) > 1).sort_values(by=["ProjectID"])

,InfraYear,Region,Province,Municipality,ImplementingOffice,ProjectID,ProjectDescription,ProjectComponentID,ProjectComponentDescription,Program,...,Editor,FundingYear,LegislativeDistrict,DistrictEngineeringOffice,GlobalID,ABC_String,ContractCost_String,CompletionDateActual,StartDate,adm3_psgc


In [29]:
# Gettings relevant columns
relevant_columns = [
    "adm3_psgc",
    "CompletionDateActual",
    "ProjectDescription",
    "ProjectComponentDescription",
    "ContractCost",
]

final_fcp_gdf = matched_fcp_gdf.loc[
    matched_fcp_gdf.adm3_psgc.notna(),
    relevant_columns + ["ProjectID"], # using ProjectID as a PKey prevent unintentional side-effects
].drop_duplicates()
final_fcp_gdf

,adm3_psgc,CompletionDateActual,ProjectDescription,ProjectComponentDescription,ContractCost,ProjectID
0,105503000,2025-05-14,"Rehabilitation of Flood Mitigation Structure, ...",Rehabilitation / Major Repair of Flood Control...,4850385.71,P00941153LZ
1,1731500000,2025-05-19,"Construction of Slope Protection Structure, In...",Construction of Flood Mitigation Structure - C...,14669999.48,P00921211LZ
2,105503000,2025-05-21,"Construction of Line Canal, Barangay Quibuar, ...",Construction of Flood Mitigation Structure - C...,4850666.92,P00941152LZ
3,306916000,2025-05-22,Construction of Slope Protection Structure alo...,Construction of Slope Protection Structure - C...,14229719.67,P00921064LZ
6,306916000,2025-03-19,Construction of Flood Control Structure along ...,Construction of Flood Mitigation Structure - C...,24125000.00,P00941410LZ
...,...,...,...,...,...,...
9178,1102319000,2022-12-24,"Construction of Bank Protection, protecting Da...",Construction of Flood Mitigation Structure - C...,76200000.00,P00620055MN
9180,1130700000,2022-10-30,Construction of Drainage System at Dr. Santiag...,Construction of Drainage Structure - Construct...,4950000.00,P00631099MN
9181,1130700000,2022-07-26,Construction of Drainage System at Amparo Home...,Construction of Drainage Structure - Construct...,9900000.00,P00631097MN
9182,1130700000,2023-03-31,Construction of Flood Mitigation Structure alo...,Construction of Flood Mitigation Structure - C...,49000000.00,P00630601MN


In [30]:
fcp_folder = "flood_control_projects"
final_fcp_gdf[relevant_columns].to_csv(
    os.path.join(data_path, fcp_folder, "flood_control_per_city.csv"),
    index=False
)